### Sample Code to train and Deploy Time Series Forecast Model

##### Install Snowpark

In [ ]:
!pip install snowflake-snowpark-python[pandas]==0.10.0

##### Install prophet model on your local instance

In [ ]:
!pip install prophet

##### Import snowpark libraries and other required libraries

In [93]:
# Snowpark
import snowflake.snowpark
from snowflake.snowpark.functions import sproc
import snowflake.snowpark.types as T
from snowflake.snowpark.session import Session
from snowflake.snowpark import version as v
import json
from config_template import snowflake_conn_prop_local as snowflake_udf_conn_prop

import pandas as pd
import numpy as np
import datetime
import io

##### Create Snowflake connection

In [96]:
session = Session.builder.configs(snowflake_udf_conn_prop).create()
print(session.sql('select current_account(), current_warehouse(), current_database(), current_schema()').collect())

session.add_packages('snowflake-snowpark-python', 'pandas', 'numpy', 'statsmodels', 'matplotlib','joblib','prophet')

[Row(CURRENT_ACCOUNT()='ANA95816', CURRENT_WAREHOUSE()='DEMO_QAS_WH', CURRENT_DATABASE()='FSI_SOLUTION', CURRENT_SCHEMA()='PUBLIC')]


The version of package joblib in the local environment is 1.2.0, which does not fit the criteria for the requirement joblib. Your UDF might not work when the package version is different between the server and your local environment
The version of package prophet in the local environment is 1.1.2, which does not fit the criteria for the requirement prophet. Your UDF might not work when the package version is different between the server and your local environment


##### Create internal stage to save model files and dependencies

In [97]:
session.sql('create or replace stage models').collect()

[Row(status='Stage area MODELS successfully created.')]

##### Define and register stored proc for prophet model

In [98]:
import pandas as pd
from prophet import Prophet
import os
from joblib import dump

@sproc(name="prophet_forecast_occ", 
       is_permanent=True, 
       stage_location="@models", 
       replace=True, 
       packages=["snowflake-snowpark-python", "prophet","joblib"])
def prophet_forecast_occ(session: snowflake.snowpark.Session, 
                         raw_table: str, 
                         out_table_name: str, 
                         stock: str) -> T.Variant:
    import pandas as pd
    from prophet import Prophet
    import os
    from joblib import dump
    df = session.sql('SELECT "Date" as "ds", "Close" as "y" from '+raw_table+' WHERE "Stock" = \''+ stock + '\' ').to_pandas()
    model = Prophet()
    model.fit(df)
    future = model.make_future_dataframe(periods=100, include_history=True)
    future.tail()
    forecast = model.predict(future)
    forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()
    df['forecast'] = forecast['yhat']
    df1 = pd.concat([forecast, df], axis=1)[["ds", "y", "yhat"]]
    df1.columns = ['Date','ds', 'actual', 'predicted']
    snowpark_df = session.write_pandas(df1, out_table_name, auto_create_table=True, overwrite=True)
    
    se = np.square(forecast.loc[:, 'yhat'] - df['y'])
    mse = np.mean(se)
    rmse = np.sqrt(mse)
    
    df_metrics =  pd.DataFrame({'model':'Prophet','se':se,'mse':mse,'rmse':rmse})

    session.create_dataframe(df_metrics).write.mode("overwrite").save_as_table("timeseries_frct_metrics")
    
    # Upload trained model to a stage
    model_file = os.path.join('/tmp', 'prophet.joblib')
    dump(model, model_file)
    session.file.put(model_file, "@models",overwrite=True)
    
    return rmse

The version of package prophet in the local environment is 1.1.2, which does not fit the criteria for the requirement prophet. Your UDF might not work when the package version is different between the server and your local environment
The version of package joblib in the local environment is 1.2.0, which does not fit the criteria for the requirement joblib. Your UDF might not work when the package version is different between the server and your local environment


##### Call the Stored Proc to train the timeseries forecast prophet model and predict

In [99]:
prophet_forecast_occ('yahoo_finance_3y_stock_data',
                 'yahoo_finance_3y_stock_data_prophet',
                 'AAPL')

'7.189412050275242'

##### Check the temp table output executed from sproc where yhat is compared to yact for the prophet model metrics

In [100]:
session.sql('select * from timeseries_frct_metrics').show()

-------------------------------------------------------------------------
|"model"  |"se"                |"mse"               |"rmse"             |
-------------------------------------------------------------------------
|Prophet  |232.33569956736463  |51.687645628642855  |7.189412050275242  |
|Prophet  |92.95862795872      |51.687645628642855  |7.189412050275242  |
|Prophet  |106.71336891933562  |51.687645628642855  |7.189412050275242  |
|Prophet  |109.00556080428768  |51.687645628642855  |7.189412050275242  |
|Prophet  |154.2271314380829   |51.687645628642855  |7.189412050275242  |
|Prophet  |90.96427639551358   |51.687645628642855  |7.189412050275242  |
|Prophet  |9.837475346128654   |51.687645628642855  |7.189412050275242  |
|Prophet  |0.3658098012746088  |51.687645628642855  |7.189412050275242  |
|Prophet  |0.9691833519163464  |51.687645628642855  |7.189412050275242  |
|Prophet  |33.82841299877501   |51.687645628642855  |7.189412050275242  |
--------------------------------------

##### Check the model file saved in internal stage model executed from the sproc 

In [101]:
session.sql("list @models").collect()

[Row(name='models/prophet.joblib.gz', size=645680, md5='d78a5368cf6df30ad524934f35a26dbb', last_modified='Mon, 3 Apr 2023 06:41:08 GMT')]

##### Now you can close the session

In [102]:
session.close()